In [1]:
!pip install -U langchain langchain-community langchain-core
!pip install -U transformers huggingface_hub
!pip install -U langsmith
!pip install -U accelerate
!pip install -U sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "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"
os.environ["LANGCHAIN_PROJECT"] = "AI-Resume-Screener"

print("LangSmith tracing enabled")

LangSmith tracing enabled


In [40]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=128,
    temperature=0.2,
    do_sample=False,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [41]:
job_description = """
We are hiring a Data Scientist.

Required Skills:
Python
Machine Learning
Pandas
NumPy
Scikit-learn
Data Visualization
SQL

Experience: 2+ years
"""

In [42]:
strong_resume = """
John Doe
Experience: 3 years as Data Scientist
Skills: Python, Machine Learning, Pandas, NumPy, Scikit-learn, SQL, Matplotlib
Tools: Jupyter, Git
"""

average_resume = """
Jane Smith
Experience: 1 year Data Analyst
Skills: Python, Pandas, Excel
Tools: Tableau
"""

weak_resume = """
Tom
Experience: Fresher
Skills: MS Word, Communication
"""

In [43]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import HuggingFacePipeline

extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an AI Resume Parser.

Extract the following:
- Skills
- Tools
- Experience (in years)

Rules:
- Only extract what is explicitly mentioned
- Do NOT assume anything

Resume:
{resume}

Output format (JSON):
{{
  "skills": [],
  "tools": [],
  "experience": ""
}}
"""
)

In [44]:
match_prompt = PromptTemplate(
    input_variables=["skills", "job"],
    template="""
Compare candidate skills with job description.

Candidate JSON:
{skills}

Job Description:
{job}

Return ONLY JSON:
{{
"matched_skills": []
}}
"""
)

In [45]:
score_prompt = PromptTemplate(
    input_variables=["match"],
    template="""
Matched skills:
{match}

Give only a score between 0 and 100.
"""
)

In [46]:
explain_prompt = PromptTemplate(
    input_variables=["score", "match"],
    template="""
Score: {score}
Matched: {match}

Return short explanation.
"""
)

In [47]:
parser = StrOutputParser()

extract_chain = extract_prompt | llm | parser
match_chain = match_prompt | llm | parser
score_chain = score_prompt | llm | parser
explain_chain = explain_prompt | llm | parser

In [52]:
def evaluate_resume(resume):

    extracted = resume

    required_skills = [
        "python","machine learning","pandas",
        "numpy","scikit-learn","sql","data visualization"
    ]

    matched = []
    for skill in required_skills:
        if skill.lower() in resume.lower():
            matched.append(skill)

    score = int((len(matched) / len(required_skills)) * 100)

    # deterministic explanation
    explanation = (
        f"Candidate matched {len(matched)} out of {len(required_skills)} required skills. "
        f"Overall suitability score is {score}%."
    )

    return {
        "Extracted": extracted,
        "Matched": matched,
        "Score": score,
        "Explanation": explanation
    }

In [54]:
print("STRONG CANDIDATE")
print(evaluate_resume(strong_resume))

print("\nAVERAGE CANDIDATE")
print(evaluate_resume(average_resume))

print("\nWEAK CANDIDATE")
print(evaluate_resume(weak_resume))

STRONG CANDIDATE
{'Extracted': '\nJohn Doe\nExperience: 3 years as Data Scientist\nSkills: Python, Machine Learning, Pandas, NumPy, Scikit-learn, SQL, Matplotlib\nTools: Jupyter, Git\n', 'Matched': ['python', 'machine learning', 'pandas', 'numpy', 'scikit-learn', 'sql'], 'Score': 85, 'Explanation': 'Candidate matched 6 out of 7 required skills. Overall suitability score is 85%.'}

AVERAGE CANDIDATE
{'Extracted': '\nJane Smith\nExperience: 1 year Data Analyst\nSkills: Python, Pandas, Excel\nTools: Tableau\n', 'Matched': ['python', 'pandas'], 'Score': 28, 'Explanation': 'Candidate matched 2 out of 7 required skills. Overall suitability score is 28%.'}

WEAK CANDIDATE
{'Extracted': '\nTom\nExperience: Fresher\nSkills: MS Word, Communication\n', 'Matched': [], 'Score': 0, 'Explanation': 'Candidate matched 0 out of 7 required skills. Overall suitability score is 0%.'}


In [55]:
#debudding 
evaluate_resume("This candidate knows cooking and dancing")

{'Extracted': 'This candidate knows cooking and dancing',
 'Matched': [],
 'Score': 0,
 'Explanation': 'Candidate matched 0 out of 7 required skills. Overall suitability score is 0%.'}